<a href="https://colab.research.google.com/github/revo-off/YOLO-CPVision/blob/main/notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

![YOLO Illustration](https://cdn.prod.website-files.com/614c82ed388d53640613982e/65390c0119ee1e54a61cac91_evolution-yolo.webp)

---
# Installation & Configuration

In [ ]:
# Install necessary libraries
# ultralytics   : YOLO26 and all computer vision tools
# gradio        : Interactive web interface
# opencv-python : Image processing (reading, drawing, conversion)
# Pillow        : Image manipulation (PIL)
# matplotlib    : Visualization and plotting
# pandas        : Data manipulation (training curves)
# numpy         : Numerical computation
# torch         : PyTorch (deep learning engine)
# torchvision   : Vision utilities for PyTorch
# seaborn       : Advanced statistical visualization
# requests      : Downloading test images
# tqdm          : Progress bars
# pyyaml        : Reading .yaml configuration files

!pip install ultralytics gradio opencv-python-headless Pillow matplotlib pandas numpy torch torchvision seaborn requests tqdm pyyaml --quiet

import importlib
packages = [
    ('ultralytics', 'ultralytics'),
    ('gradio', 'gradio'),
    ('cv2', 'opencv-python'),
    ('PIL', 'Pillow'),
    ('matplotlib', 'matplotlib'),
    ('pandas', 'pandas'),
    ('numpy', 'numpy'),
    ('torch', 'torch'),
    ('torchvision', 'torchvision'),
    ('seaborn', 'seaborn'),
    ('yaml', 'pyyaml'),
]

print('\n Checking installed packages :')
all_ok = True
for module_name, package_name in packages:
    try:
        mod = importlib.import_module(module_name)
        version = getattr(mod, '__version__', 'OK')
        print('✅ ' + package_name.ljust(25) + ' v' + str(version))
    except ImportError:
        print('❌ ' + package_name.ljust(25) + ' NOT INSTALLED')
        all_ok = False

print()
if all_ok:
    print('✅ All packages are installed! You can continue.')
else:
    print('⚠️ Some packages are missing. Please restart this cell.')

---
# Dataprep

In [ ]:
from ultralytics.utils import DATASETS_DIR
from ultralytics.data.utils import check_det_dataset

print("📥 COCO Downloading...")
dataset_info = check_det_dataset('coco.yaml')


print(f"\n📁 Dataset downloaded to: {DATASETS_DIR}")
print(f"\n📊 Information about the dataset :")
print(f"   - Number of classes : {dataset_info['nc']}")
print(f"   - Classes : {dataset_info['names']}")

In [ ]:
import glob
import matplotlib.pyplot as plt
from PIL import Image

train_images = glob.glob(str(DATASETS_DIR / 'coco' / 'images' / 'train2017' / '*.jpg'))

if train_images:
    fig, axes = plt.subplots(2, 4, figsize=(16, 8))
    fig.suptitle('📸 Example of images from the COCO dataset', fontsize=16, fontweight='bold')

    for i, ax in enumerate(axes.flatten()):
        if i < len(train_images):
            img = Image.open(train_images[i])
            ax.imshow(img)
            ax.set_title(f'Image {i+1}', fontsize=10)
            ax.axis('off')
        else:
            ax.axis('off')

    plt.tight_layout()
    plt.show()
    print(f"\n✅ {len(train_images)} training images found in the dataset.")
else:
    print("⚠️ Images not found. The dataset will be downloaded during training.")


# TODO : How many data points do we have in the train, val, and test sets?
print(f"\n📊 Dataset splits :")
print(f"   - Training set : {dataset_info['train']}")
print(f"   - Validation set : {dataset_info['val']}")
print(f"   - Test set : {dataset_info['test']}")

---
# Model training

YOLO26 offers several model sizes depending on your needs:

| Model | Size | Speed | Accuracy | Recommended usage |
|--------|--------|---------|-----------|------------------|
| `yolo26n` | Nano | ⚡⚡⚡⚡⚡ | ⭐⭐ | Mobile, IoT, real-time |
| `yolo26s` | Small | ⚡⚡⚡⚡ | ⭐⭐⭐ | Lightweight applications |
| `yolo26m` | Medium | ⚡⚡⚡ | ⭐⭐⭐⭐ | General use |
| `yolo26l` | Large | ⚡⚡ | ⭐⭐⭐⭐⭐ | High accuracy |
| `yolo26x` | XLarge | ⚡ | ⭐⭐⭐⭐⭐ | Research, servers |


In [ ]:
from ultralytics import YOLO

model = YOLO('yolo26m.pt')
print("✅ Model loaded!")

print("\n🚀 Starting training...")
print("   (This may take a few minutes)\n")

results = model.train(
    data='coco.yaml',       # Dataset configuration file
    epochs=500,             # Number of epochs
    imgsz=640,              # Image size
    batch=32,               # Batch size
    name='tp_yolo26',       # Experiment name
    patience=100,           # Early stopping after X epochs without improvement
    verbose=True,           # Show logs
    device=device,          # Use GPU if available
)

print("\n✅ Training completed!")
print(f"📁 Results saved in : {results.save_dir}")

---
# Model evaluation

### Main metrics:

| Metric | Description | Good value |
|----------|-------------|-------------|
| **mAP50** | Mean average precision at IoU=0.5 | > 0.5 |
| **mAP50-95** | Mean average precision over multiple IoU thresholds | > 0.3 |
| **Precision** | Among the detections, how many are correct? | > 0.7 |
| **Recall** | Among the real objects, how many are detected? | > 0.7 |

### What is IoU?

**IoU** (Intersection over Union) measures the overlap between the predicted box and the ground truth box:

```
IoU = Area(Intersection) / Area(Union)
```

- IoU = 1.0 → Perfect detection
- IoU = 0.5 → Acceptable detection
- IoU < 0.5 → Poor detection

In [ ]:
from pathlib import Path
from ultralytics import YOLO

best_model_path = Path(results.save_dir) / 'weights' / 'best.pt'
print(f"📥 Best model path: {best_model_path}")

best_model = YOLO(best_model_path)

print("\n🔍 Evaluation in progress...")
metrics = best_model.val(data='coco.yaml', verbose=True)

print("\n" + "="*50)
print("📊 Evaluation Results")
print("="*50)
print(f"  mAP50     : {metrics.box.map50:.4f}  (precision at IoU=0.5)")
print(f"  mAP50-95  : {metrics.box.map:.4f}  (global precision)")
print(f"  Precision : {metrics.box.mp:.4f}")
print(f"  Recall    : {metrics.box.mr:.4f}")
print("="*50)

map50 = metrics.box.map50
if map50 >= 0.5:
    print("✅ The model has good precision (mAP50 >= 0.5). Detections are generally correct.")
elif map50 >= 0.3:
    print("⚠️ The model has average precision. It could be improved with more data or training epochs.")
else:
    print("❌ The model has low precision (mAP50 < 0.3). Training requires significant adjustments.")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

results_csv = Path(results.save_dir) / 'results.csv'

if results_csv.exists():
    df = pd.read_csv(results_csv)
    df.columns = df.columns.str.strip()

    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    fig.suptitle('📈 YOLO26 Training vs Validation Curves', fontsize=16, fontweight='bold')

    # Box Loss
    if 'train/box_loss' in df.columns:
        axes[0, 0].plot(df['epoch'], df['train/box_loss'], 'b-', linewidth=2, label='Train', marker='o', markersize=3)
        if 'val/box_loss' in df.columns:
            axes[0, 0].plot(df['epoch'], df['val/box_loss'], 'b--', linewidth=2, label='Validation', marker='s', markersize=3)
        axes[0, 0].set_title('Box Loss', fontweight='bold')
        axes[0, 0].set_xlabel('Epoch')
        axes[0, 0].set_ylabel('Loss')
        axes[0, 0].legend()
        axes[0, 0].grid(True, alpha=0.3)

    # Classification Loss
    if 'train/cls_loss' in df.columns:
        axes[0, 1].plot(df['epoch'], df['train/cls_loss'], 'r-', linewidth=2, label='Train', marker='o', markersize=3)
        if 'val/cls_loss' in df.columns:
            axes[0, 1].plot(df['epoch'], df['val/cls_loss'], 'r--', linewidth=2, label='Validation', marker='s', markersize=3)
        axes[0, 1].set_title('Classification Loss', fontweight='bold')
        axes[0, 1].set_xlabel('Epoch')
        axes[0, 1].set_ylabel('Loss')
        axes[0, 1].legend()
        axes[0, 1].grid(True, alpha=0.3)

    # DFL Loss
    if 'train/dfl_loss' in df.columns:
        axes[0, 2].plot(df['epoch'], df['train/dfl_loss'], 'g-', linewidth=2, label='Train', marker='o', markersize=3)
        if 'val/dfl_loss' in df.columns:
            axes[0, 2].plot(df['epoch'], df['val/dfl_loss'], 'g--', linewidth=2, label='Validation', marker='s', markersize=3)
        axes[0, 2].set_title('DFL Loss', fontweight='bold')
        axes[0, 2].set_xlabel('Epoch')
        axes[0, 2].set_ylabel('Loss')
        axes[0, 2].legend()
        axes[0, 2].grid(True, alpha=0.3)

    # mAP50
    if 'metrics/mAP50(B)' in df.columns:
        axes[1, 0].plot(df['epoch'], df['metrics/mAP50(B)'], 'purple', linewidth=2, label='Validation', marker='o', markersize=3)
        axes[1, 0].set_title('mAP50', fontweight='bold')
        axes[1, 0].set_xlabel('Epoch')
        axes[1, 0].set_ylabel('mAP50')
        axes[1, 0].legend()
        axes[1, 0].grid(True, alpha=0.3)

    # Precision
    if 'metrics/precision(B)' in df.columns:
        axes[1, 1].plot(df['epoch'], df['metrics/precision(B)'], 'orange', linewidth=2, label='Validation', marker='o', markersize=3)
        axes[1, 1].set_title('Precision', fontweight='bold')
        axes[1, 1].set_xlabel('Epoch')
        axes[1, 1].set_ylabel('Precision')
        axes[1, 1].legend()
        axes[1, 1].grid(True, alpha=0.3)

    # Recall
    if 'metrics/recall(B)' in df.columns:
        axes[1, 2].plot(df['epoch'], df['metrics/recall(B)'], 'cyan', linewidth=2, label='Validation', marker='o', markersize=3)
        axes[1, 2].set_title('Recall', fontweight='bold')
        axes[1, 2].set_xlabel('Epoch')
        axes[1, 2].set_ylabel('Recall')
        axes[1, 2].legend()
        axes[1, 2].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()
    print("✅ Curves displayed!")
else:
    print("⚠️ Results file not found.")

---
# Deployement on Gradio

In [ ]:
import gradio as gr
import numpy as np
import cv2
from PIL import Image
from ultralytics import YOLO

try:
    best_model
except NameError:
    best_model_path = 'runs/detect/tp_yolo26/weights/best.pt'
    best_model = YOLO(best_model_path)

def detecter_objets(image, seuil_confiance=0.25):
    if image is None:
        return None, "⚠️ En attente du flux vidéo..."

    resultats = best_model.predict(
        source=image,
        conf=seuil_confiance,
        iou=0.45,
        verbose=False
    )

    result = resultats[0]
    image_annotee = result.plot()

    image_annotee_rgb = cv2.cvtColor(image_annotee, cv2.COLOR_BGR2RGB)

    nb_detections = len(result.boxes)
    texte_resultats = f"🎯 {nb_detections} object(s) detected"

    return Image.fromarray(image_annotee_rgb), texte_resultats

with gr.Blocks() as demo:
    gr.Markdown("# YOLO - Real-Time Object Detection")

    with gr.Row():
        with gr.Column():
            input_img = gr.Image(
                label="Camera Feed",
                sources=["webcam"],
                type="pil",
                streaming=True
            )

            seuil = gr.Slider(0.1, 0.9, value=0.25, label="Confidence Threshold")

        with gr.Column():
            output_img = gr.Image(label="YOLO Detection", type="pil")
            output_txt = gr.Textbox(label="Statistics")

    input_img.stream(
        fn=detecter_objets,
        inputs=[input_img, seuil],
        outputs=[output_img, output_txt]
    )

demo.launch(share=True)